In [1]:
import pandas as pd

training_start_date = '2020-01-01'

# Load data
train_df = pd.read_csv('~/data/Germany_time_zone/gas/train_with_gas.csv')

# Filter train_df to include only data from 2020 onward
train_df = train_df[pd.to_datetime(train_df['ds']) >= pd.Timestamp(training_start_date)]

# Convert to UTC while handling DST ambiguities
train_df['ds'] = pd.to_datetime(train_df['ds']).dt.tz_localize('Europe/Berlin', ambiguous='infer').dt.tz_convert('UTC')

# Check date ranges
print("First date in train_df:", train_df['ds'].min())
print("Last date in train_df:", train_df['ds'].max())

stat_exog_list = []

hist_exog_list = ['month_sin', 'month_cos', 'day_of_week_sin', 'day_of_week_cos',
                  'hour_sin', 'hour_cos', 'is_holiday', 'is_weekend']

futr_exog_list = ['month_sin', 'month_cos', 'day_of_week_sin', 'day_of_week_cos',
                  'hour_sin', 'hour_cos', 'is_holiday', 'is_weekend']

First date in train_df: 2019-12-31 23:00:00+00:00
Last date in train_df: 2023-12-31 22:00:00+00:00


In [2]:
train_df.head()

,ds,y,unique_id,day_of_week,month,hour,is_weekend,is_holiday,month_sin,month_cos,day_of_week_sin,day_of_week_cos,hour_sin,hour_cos,gas_price
10969,2019-12-31 23:00:00+00:00,41.88,electricity_prices,2,1,0,0,1,0.5,0.866025,0.974928,-0.222521,0.000000,1.000000,13.218
10970,2020-01-01 00:00:00+00:00,38.60,electricity_prices,2,1,1,0,1,0.5,0.866025,0.974928,-0.222521,0.258819,0.965926,13.218
10971,2020-01-01 01:00:00+00:00,36.55,electricity_prices,2,1,2,0,1,0.5,0.866025,0.974928,-0.222521,0.500000,0.866025,13.218
10972,2020-01-01 02:00:00+00:00,32.32,electricity_prices,2,1,3,0,1,0.5,0.866025,0.974928,-0.222521,0.707107,0.707107,13.218
10973,2020-01-01 03:00:00+00:00,30.85,electricity_prices,2,1,4,0,1,0.5,0.866025,0.974928,-0.222521,0.866025,0.500000,13.218


Hyperparameter Optimisation

In [ ]:
from neuralforecast import NeuralForecast
from neuralforecast.auto import AutoNHITS
import optuna

horizon = 24
historical = 24 * 7
validation_size = int(0.1 * len(train_df))
# validation_size = 24 * 30 * 3 # 3 months 

number_of_samples = 20
model_name = 'AutoNHITS'

def config_nhits(trial):
    h = horizon
    return {
        "stat_exog_list": trial.suggest_categorical("stat_exog_list", [stat_exog_list]),
        "hist_exog_list": trial.suggest_categorical("hist_exog_list", [hist_exog_list]),
        "futr_exog_list": trial.suggest_categorical("futr_exog_list", [futr_exog_list]),
        "mlp_units": trial.suggest_categorical(
            "mlp_units",
            [
                [[512, 512], [512, 512],
        [512, 512]],
                [[1024, 1024], [1024, 1024],
        [1024, 1024]],
            ]
        ),
        "input_size": trial.suggest_categorical(
            "input_size",
            # [x * 24 for x in [1, 3, 4, 7, 10, 14]]
            [historical]
        ),
        "max_steps": trial.suggest_int("max_steps", 1000, 10000, step=1000),  # Maximum steps (quantized by 1000)
        "step_size": trial.suggest_categorical("step_size",
                                               [1, h]
                                               # [h]
                                               ),
        "n_pool_kernel_size": trial.suggest_categorical(
            "n_pool_kernel_size",
            [
                [2, 2, 1],
                [1, 1, 1],    # 3 * [1]
                [2, 2, 2],    # 3 * [2]
                [4, 4, 4],    # 3 * [4]
                [8, 4, 1],
                [16, 8, 1],
            ]
        ),  # MaxPool kernel sizes
        "n_freq_downsample": trial.suggest_categorical(
            "n_freq_downsample",
            [
                [168, 24, 1],
                [24, 12, 1],
                [180, 60, 1],
                [60, 8, 1],
                [40, 20, 1],
                [1, 1, 1],
                [4, 2, 1],
            ]
        ),  # Frequency downsampling ratios
        "learning_rate": trial.suggest_loguniform("learning_rate", 1e-4, 1e-1),  # Learning rate (log-uniform)
        "scaler_type": trial.suggest_categorical("scaler_type", [None, "robust", "standard"]),  # Scaler type
        "batch_size": trial.suggest_categorical("batch_size", [32, 64, 128, 256]),  # Batch size
        "windows_batch_size": trial.suggest_categorical("windows_batch_size", [128, 256, 512, 1024]),  # Windows batch size
        "loss": None,  # Fixed parameter
        "random_seed": trial.suggest_int("random_seed", 1, 20),  # Random seed
    }
    
model = AutoNHITS(h=horizon,
                  config=config_nhits,
                  search_alg=optuna.samplers.TPESampler(),
                  backend='optuna',
                  num_samples=number_of_samples,
                 )


# Train the model on the entire training data just once
nf = NeuralForecast(models=[model], freq='h')
nf.fit(df=train_df, val_size=validation_size) 

[I 2025-02-13 11:30:55,242] A new study created in memory with name: no-name-081a016b-6858-400d-aa18-b5298b36e56f
/pfs/data5/home/kit/iai/fn8011/ml/lib64/python3.9/site-packages/optuna/distributions.py:524: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains [] which is of type list.
  warnings.warn(message)
/pfs/data5/home/kit/iai/fn8011/ml/lib64/python3.9/site-packages/optuna/distributions.py:524: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains ['month_sin', 'month_cos', 'day_of_week_sin', 'day_of_week_cos', 'hour_sin', 'hour_cos', 'is_holiday', 'is_weekend'] which is of type list.
  warnings.warn(message)
/pfs/data5/home/kit/iai/fn8011/ml/lib64/python3.9/site-packages/optuna/distributions.py:524: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for per

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

In [ ]:
results = nf.models[0].results.trials_dataframe()
results.drop(columns='user_attrs_ALL_PARAMS')


In [ ]:
# # using raytune:
# nf.models[0].results.get_best_result().config


nf.models[0].results.best_trial.user_attrs["ALL_PARAMS"]

In [ ]:
# # Ray:
# best_config = nf.models[0].results.get_best_result().config

# Optuna
best_config = nf.models[0].results.best_trial.user_attrs["ALL_PARAMS"]

In [ ]:
import json

# Let's say you have your best params dictionary
best_params = nf.models[0].results.best_trial.user_attrs["ALL_PARAMS"]

# Remove non-serializable keys (e.g., the loss objects)
best_params.pop('loss', None)
best_params.pop('valid_loss', None)

# Save it to a JSON file
with open("../best/best_config_basic.json", "w") as f:
    json.dump(best_params, f, indent=4)

In [ ]:
print('done')